# Museum Analytics: Visitor-Population Correlation Analysis

This notebook demonstrates the analysis of museum visitor data and its correlation with city population using linear regression.

In [ ]:
# Import required libraries
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Add the src directory to the path
sys.path.append('/workspace/museum_analytics/src')

# Import our custom modules
from data_extraction.wikipedia_scraper import WikipediaMuseumScraper
from data_extraction.population_data import PopulationDataExtractor
from data_processing.harmonizer import DataHarmonizer
from models.regression_model import MuseumRegressionModel
from database.db_manager import DatabaseManager

print("Libraries imported successfully!")

In [ ]:
# Extract museum data from Wikipedia
print("Extracting museum data from Wikipedia...")
museum_scraper = WikipediaMuseumScraper()
museums = museum_scraper.extract_museum_data()

if museums:
    museum_df = museum_scraper.save_to_csv(museums, "/workspace/museum_data.csv")
    print(f"Successfully extracted {len(museums)} museums")
    print("\nSample museum data:")
    print(museum_df.head())
else:
    print("Failed to extract museum data")

In [ ]:
# Extract population data
print("Extracting city population data...")
population_extractor = PopulationDataExtractor()

if museums:
    unique_cities = list(set([museum['city'] for museum in museums]))
    population_data = population_extractor.get_city_population_data(unique_cities)
    
    if population_data:
        population_df = population_extractor.save_to_csv(population_data, "/workspace/city_population.csv")
        print(f"Successfully extracted population data for {len(population_data)} cities")
        print("\nSample population data:")
        print(population_df.head())
    else:
        print("Failed to extract population data")
else:
    print("No museum data available for population extraction")

In [ ]:
# Harmonize the data
print("Harmonizing museum and population data...")
harmonizer = DataHarmonizer("/workspace/museum_analytics.db")
harmonized_df = harmonizer.harmonize_data("/workspace/museum_data.csv", "/workspace/city_population.csv")

if not harmonized_df.empty:
    print(f"Successfully harmonized {len(harmonized_df)} museum records")
    print("\nSample harmonized data:")
    print(harmonized_df.head())
else:
    print("Failed to harmonize data")

In [ ]:
# Train the regression model
if not harmonized_df.empty:
    print("Training linear regression model...")
    model = MuseumRegressionModel("/workspace/museum_regression_model.pkl")
    
    # Train the model
    metrics = model.train("/workspace/harmonized_museum_data.csv")
    
    if 'error' not in metrics:
        print("\nModel Training Results:")
        print(f"Training R²: {metrics['train_r2']:.4f}")
        print(f"Test R²: {metrics['test_r2']:.4f}")
        print(f"Correlation: {metrics['correlation']:.4f}")
        print(f"Training MSE: {metrics['train_mse']:,.2f}")
        print(f"Test MSE: {metrics['test_mse']:,.2f}")
        
        # Get model summary
        summary = model.get_model_summary()
        print(f"\nModel Equation: {summary['equation']}")
    else:
        print(f"Error training model: {metrics['error']}")
else:
    print("No data available for model training")

In [ ]:
# Create visualizations
if not harmonized_df.empty and 'error' not in metrics:
    print("Creating visualizations...")
    
    # Load data for plotting
    df = pd.read_csv("/workspace/harmonized_museum_data.csv")
    df_clean = df.dropna(subset=['city_population', 'annual_visitors'])
    
    # Create scatter plot with regression line
    plt.figure(figsize=(12, 8))
    plt.scatter(df_clean['city_population'], df_clean['annual_visitors'], 
               alpha=0.6, color='blue', label='Actual Data')
    
    # Create regression line
    X = np.linspace(df_clean['city_population'].min(), df_clean['city_population'].max(), 100)
    y_pred = model.predict(X)
    plt.plot(X, y_pred, 'r-', linewidth=2, label='Regression Line')
    
    plt.xlabel('City Population')
    plt.ylabel('Annual Visitors')
    plt.title('Museum Visitors vs City Population - Linear Regression Model')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Create model visualizations
    model.create_visualizations("/workspace/harmonized_museum_data.csv", "/workspace/plots")
    print("Visualizations saved to '/workspace/plots/' directory")
else:
    print("No model available for visualization")

In [ ]:
# Test model predictions
if not harmonized_df.empty and 'error' not in metrics:
    print("Model Predictions for Different City Sizes:")
    print("=" * 50)
    
    test_cities = [
        (100_000, "Small City"),
        (500_000, "Medium City"),
        (1_000_000, "Large City"),
        (5_000_000, "Metropolitan Area"),
        (10_000_000, "Mega City")
    ]
    
    for population, city_type in test_cities:
        prediction = model.predict(population)
        print(f"{city_type} ({population:,} people): {prediction:,.0f} predicted visitors")
    
    print("\n" + "=" * 50)
    print("Note: These are predictions based on city population only.")
    print("Actual museum attendance depends on many other factors.")
else:
    print("No trained model available for predictions")